# CSQA compositional data pipeline validation

This notebook validates the full compositional data-prep path for **CommonsenseQA / CSQA** starting from the fine-grained MCTS routing output and ending at the exact structures consumed by `training/train_compositional_router.py`.

It is intentionally sequential and verbose. Each section checks one genuine processing step:

1. Fine-grained MCTS JSONL and residual tensors.
2. Per-row canonicalization into edit-program records.
3. Directory-level canonicalization.
4. Primitive and pair support aggregation.
5. Primitive catalogue filtering (`O_train`).
6. Legal-program enumeration (`E_legal`).
7. Incidence and pair-incidence tensors.
8. Observed candidate projection from canonical programs to legal rows.
9. `load_artifacts`, `CompositionalDataset`, and `collate_compositional`.
10. Optional dense-delta materialization for dense-supervision controls.

By default the notebook writes rebuilt artifacts under `notebooks/_compositional_csqa_validation_artifacts/` and leaves the checked-in run directories untouched. Set `RUN_FULL_PIPELINE = False` in the setup cell to only inspect the existing CSQA artifacts.

## 1. Setup and path resolution

The notebook is anchored at the repo root. `FINE_MCTS_DIR` should point at the original fine-grained MCTS output directory containing `commonsenseqa.jsonl`, `config.json`, and residual `.pt` files. The defaults match the current CSQA compositional run if those paths exist on this machine.

`RUN_FULL_PIPELINE` controls whether we rebuild into a scratch directory. When it is `False`, later sections inspect the existing canonical and compositional outputs.

In [1]:
from __future__ import annotations

import json
import math
import os
import shutil
import sys
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint
from typing import Any, Dict, Iterable, List, Sequence

import torch
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BENCH = "commonsenseqa"
EXISTING_CANONICAL_DIR = REPO_ROOT / "compositional_runs/csqa_canonical"
EXISTING_CATALOGUE_DIR = REPO_ROOT / "compositional_runs/csqa_compositional"

# Pull the original MCTS directory from the canonical config when available.
existing_cfg_path = EXISTING_CANONICAL_DIR / "config.json"
existing_cfg = json.loads(existing_cfg_path.read_text()) if existing_cfg_path.is_file() else {}
default_fine_dir = (
    existing_cfg.get("canonicalization", {}).get("source_data_dir")
    or "fine_routing_data_commonsenseqa_mcts"
)

FINE_MCTS_DIR = Path(os.environ.get("FINE_MCTS_DIR", default_fine_dir)).expanduser()
if not FINE_MCTS_DIR.is_absolute():
    FINE_MCTS_DIR = REPO_ROOT / FINE_MCTS_DIR

SCRATCH_ROOT = REPO_ROOT / "notebooks/_compositional_csqa_validation_artifacts"
SCRATCH_CANONICAL_DIR = SCRATCH_ROOT / "csqa_canonical"
SCRATCH_CATALOGUE_DIR = SCRATCH_ROOT / "csqa_compositional"
SCRATCH_DENSE_PT = SCRATCH_ROOT / "commonsenseqa_dense_from_canonical.pt"

RUN_FULL_PIPELINE = os.environ.get("RUN_FULL_PIPELINE", "1") != "0"
CLEAN_SCRATCH_FIRST = os.environ.get("CLEAN_SCRATCH_FIRST", "0") == "1"
USE_UNIQUE_SUPPORT_PER_QUESTION = os.environ.get("COUNT_UNIQUE_PER_QUESTION", "0") == "1"

# Match the existing CSQA example unless you intentionally override.
FILTER_MIN_COUNT = int(os.environ.get("FILTER_MIN_COUNT", "50"))
FILTER_MIN_QUESTIONS = int(os.environ.get("FILTER_MIN_QUESTIONS", "50"))
FILTER_MIN_BENCHMARKS = int(os.environ.get("FILTER_MIN_BENCHMARKS", "1"))
KEEP_KINDS = tuple(os.environ.get("KEEP_KINDS", "skip,repeat,swap").split(","))

if RUN_FULL_PIPELINE and CLEAN_SCRATCH_FIRST and SCRATCH_ROOT.exists():
    shutil.rmtree(SCRATCH_ROOT)
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)

CANONICAL_DIR = SCRATCH_CANONICAL_DIR if RUN_FULL_PIPELINE else EXISTING_CANONICAL_DIR
CATALOGUE_DIR = SCRATCH_CATALOGUE_DIR if RUN_FULL_PIPELINE else EXISTING_CATALOGUE_DIR

print("REPO_ROOT:", REPO_ROOT)
print("FINE_MCTS_DIR:", FINE_MCTS_DIR, "exists=", FINE_MCTS_DIR.is_dir())
print("CANONICAL_DIR:", CANONICAL_DIR)
print("CATALOGUE_DIR:", CATALOGUE_DIR)
print("RUN_FULL_PIPELINE:", RUN_FULL_PIPELINE)
print("support thresholds:", FILTER_MIN_COUNT, FILTER_MIN_QUESTIONS, FILTER_MIN_BENCHMARKS, KEEP_KINDS)

REPO_ROOT: /home/janerik/flexible-test-time-program-selection
FINE_MCTS_DIR: /home/janerik/generalized_transformer-2/dr-llm/fine_routing_data_commonsenseqa_mcts exists= True
CANONICAL_DIR: /home/janerik/flexible-test-time-program-selection/notebooks/_compositional_csqa_validation_artifacts/csqa_canonical
CATALOGUE_DIR: /home/janerik/flexible-test-time-program-selection/notebooks/_compositional_csqa_validation_artifacts/csqa_compositional
RUN_FULL_PIPELINE: True
support thresholds: 50 50 1 ('skip', 'repeat', 'swap')


## 2. Small inspection helpers

These helpers avoid hiding structure behind pandas. Most checks assert invariants used downstream by the data prep or trainer, so a failure points to the first broken contract.

In [2]:
def read_jsonl(path: Path, limit: int | None = None) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            if limit is not None and len(rows) >= limit:
                break
    return rows


def count_jsonl(path: Path) -> int:
    with open(path) as f:
        return sum(1 for line in f if line.strip())


def first_jsonl(path: Path) -> Dict[str, Any]:
    rows = read_jsonl(path, limit=1)
    if not rows:
        raise ValueError(f"empty JSONL: {path}")
    return rows[0]


def shape_of(x: Any) -> Any:
    if torch.is_tensor(x):
        return {"shape": tuple(x.shape), "dtype": str(x.dtype), "device": str(x.device)}
    if isinstance(x, list):
        return {"type": "list", "len": len(x), "first_shape": shape_of(x[0]) if x else None}
    if isinstance(x, dict):
        return {k: shape_of(v) for k, v in x.items()}
    return type(x).__name__


def summarize_numeric(xs: Sequence[float]) -> Dict[str, float]:
    vals = [float(x) for x in xs]
    if not vals:
        return {"n": 0}
    vals_sorted = sorted(vals)
    def q(p: float) -> float:
        idx = min(len(vals_sorted) - 1, max(0, int(round(p * (len(vals_sorted) - 1)))))
        return vals_sorted[idx]
    return {
        "n": len(vals),
        "min": min(vals),
        "p25": q(0.25),
        "median": q(0.50),
        "p75": q(0.75),
        "max": max(vals),
        "mean": sum(vals) / len(vals),
    }


def key_hist(rows: Sequence[Dict[str, Any]], key: str) -> Counter:
    c: Counter = Counter()
    for row in rows:
        value = row.get(key)
        if isinstance(value, list):
            c[len(value)] += 1
        else:
            c[value] += 1
    return c


def assert_contiguous_indices(rows: Sequence[Dict[str, Any]], field: str = "idx") -> None:
    got = [int(r[field]) for r in rows]
    expected = list(range(len(rows)))
    assert got == expected, f"{field} not contiguous: first mismatch near {next((i for i, (a, b) in enumerate(zip(got, expected)) if a != b), 'end')}"


def display_dict(title: str, d: Dict[str, Any]) -> None:
    display(Markdown(f"### {title}"))
    pprint(d, sort_dicts=False)


def compact_row(row: Dict[str, Any], *, max_programs: int = 5, max_explored: int = 5) -> Dict[str, Any]:
    out = dict(row)
    if "programs" in out:
        out["programs"] = out["programs"][:max_programs]
        out["programs_truncated_to"] = max_programs
    if "explored" in out:
        out["explored"] = out["explored"][:max_explored]
        out["explored_truncated_to"] = max_explored
    if "router_target" in out:
        out["router_target"] = out["router_target"][:max_explored]
        out["router_target_truncated_to"] = max_explored
    return out

## 3. Fine-grained MCTS output: file-level contract

This is the source format produced by the fine-routing data prep. In MCTS mode each row should carry an anchor sequence, explored candidate layer sequences, per-candidate scores/deltas, optional router-target mass, and residual tensors aligned by JSONL row index.

In [3]:
fine_jsonl = FINE_MCTS_DIR / f"{BENCH}.jsonl"
fine_config_path = FINE_MCTS_DIR / "config.json"
fine_pivot_residuals_path = FINE_MCTS_DIR / f"{BENCH}_pivot_residuals.pt"
fine_full_residuals_path = FINE_MCTS_DIR / f"{BENCH}_full_residuals.pt"

required_fine_paths = [fine_jsonl, fine_config_path, fine_pivot_residuals_path]
missing = [str(p) for p in required_fine_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing fine-grained MCTS inputs:\n" + "\n".join(missing))

fine_config = json.loads(fine_config_path.read_text())
fine_n_rows = count_jsonl(fine_jsonl)
fine_sample_rows = read_jsonl(fine_jsonl, limit=128)
fine_first = fine_sample_rows[0]

print("fine JSONL:", fine_jsonl)
print("rows:", fine_n_rows)
print("config keys:", sorted(fine_config.keys())[:20], "...")
print("pivot residuals path:", fine_pivot_residuals_path)
print("full residuals path exists:", fine_full_residuals_path.exists())

assert fine_config.get("use_mcts") is True, "CSQA source should be MCTS-mode for this notebook"
assert fine_first.get("search_mode") == "mcts" or "explored" in fine_first, "expected MCTS fields"
assert "anchor_sequence" in fine_first
assert "explored" in fine_first
assert len(fine_first["anchor_sequence"]) > 0
assert fine_n_rows > 0

display_dict("Fine MCTS config subset", {
    "model_name": fine_config.get("model_name"),
    "data_split": fine_config.get("data_split"),
    "pivot_layer": fine_config.get("pivot_layer"),
    "editable_start": fine_config.get("editable_start"),
    "max_local_edits": fine_config.get("max_local_edits"),
    "swap_radius": fine_config.get("swap_radius"),
    "mcts_num_simulations": fine_config.get("mcts_num_simulations"),
    "continuous_target_beta": fine_config.get("continuous_target_beta"),
})

fine JSONL: /home/janerik/generalized_transformer-2/dr-llm/fine_routing_data_commonsenseqa_mcts/commonsenseqa.jsonl
rows: 9287
config keys: ['benchmarks', 'coarse_router_checkpoint', 'continuous_delta_clip', 'continuous_gate_tau', 'continuous_target_beta', 'data_split', 'delta_clip', 'editable_start', 'enumerate_deviations', 'gate_batch_size', 'gate_checkpoint', 'gate_epochs', 'gate_hidden_dim', 'gate_lr', 'gate_tau', 'gate_threshold_gamma', 'gate_w0', 'gate_w1', 'gpu_rank', 'include_anchor_confidence'] ...
pivot residuals path: /home/janerik/generalized_transformer-2/dr-llm/fine_routing_data_commonsenseqa_mcts/commonsenseqa_pivot_residuals.pt
full residuals path exists: True


### Fine MCTS config subset

{'model_name': 'Qwen/Qwen2.5-0.5B-Instruct',
 'data_split': 'train',
 'pivot_layer': 16,
 'editable_start': 17,
 'max_local_edits': 2,
 'swap_radius': 3,
 'mcts_num_simulations': 250,
 'continuous_target_beta': 2.0}


## 4. Fine-grained MCTS output: row-level structure

The first row is intentionally inspected before any transformation. The compositional path only sees the route sequences and score/delta metadata; it does not train directly on the fine-router `router_target` vector.

In [4]:
display_dict("Fine first row, compact", compact_row(fine_first, max_explored=8))

explored_counts = [len(r.get("explored", [])) for r in fine_sample_rows]
target_sums = [sum(float(x) for x in r.get("router_target", [])) for r in fine_sample_rows if "router_target" in r]
best_delta_values = [float(r.get("best_delta", 0.0)) for r in fine_sample_rows if r.get("best_delta") is not None]
anchor_lengths = Counter(len(r["anchor_sequence"]) for r in fine_sample_rows)

summary = {
    "sample_rows": len(fine_sample_rows),
    "anchor_lengths": dict(anchor_lengths),
    "explored_count_summary": summarize_numeric(explored_counts),
    "router_target_sum_summary": summarize_numeric(target_sums),
    "best_delta_summary": summarize_numeric(best_delta_values),
    "first_row_explored_fields": sorted(fine_first.get("explored", [{}])[0].keys()) if fine_first.get("explored") else [],
}
display_dict("Fine MCTS row diagnostics", summary)

# Router target is a fine-router training target, not the compositional target.
# It is still useful to validate that it is aligned with explored candidates.
if "router_target" in fine_first:
    assert len(fine_first["router_target"]) == len(fine_first.get("explored", []))
    print("first row router_target sum:", sum(float(x) for x in fine_first["router_target"]))

for exp in fine_first.get("explored", [])[:5]:
    assert "seq" in exp and "delta" in exp and "score" in exp
    assert len(exp["seq"]) == len(fine_first["anchor_sequence"]), "route seq length should match anchor length"

### Fine first row, compact

{'benchmark_id': 'commonsenseqa',
 'question_id': 0,
 'question_hash': '0d118b641f7967725f863abf30f869a8',
 'anchor_sequence': [0,
                     1,
                     2,
                     3,
                     4,
                     5,
                     6,
                     7,
                     8,
                     9,
                     10,
                     11,
                     12,
                     13,
                     14,
                     15,
                     16,
                     17,
                     18,
                     17,
                     20,
                     21,
                     22,
                     22],
 'anchor_score': -4.08203125,
 'pivot_layer_index': 16,
 'search_mode': 'mcts',
 'scoring_mode': 'continuous',
 'gate_label': 1,
 'best_seq': [0,
              1,
              2,
              3,
              4,
              5,
              6,
              7,
              8,
              9,
   

### Fine MCTS row diagnostics

{'sample_rows': 128,
 'anchor_lengths': {24: 128},
 'explored_count_summary': {'n': 128,
                            'min': 154.0,
                            'p25': 177.0,
                            'median': 184.0,
                            'p75': 191.0,
                            'max': 205.0,
                            'mean': 183.875},
 'router_target_sum_summary': {'n': 128,
                               'min': 0.999999999999999,
                               'p25': 0.9999999999999997,
                               'median': 0.9999999999999999,
                               'p75': 1.0000000000000002,
                               'max': 1.0000000000000009,
                               'mean': 1.0},
 'best_delta_summary': {'n': 128,
                        'min': 0.1136009693145752,
                        'p25': 0.5791778564453125,
                        'median': 0.92724609375,
                        'p75': 1.36767578125,
                        'max': 5.0654296875

## 5. Residual tensor alignment

`CompositionalDataset` later indexes `pivot_residuals[residual_idx]`, where `residual_idx` is the canonical JSONL line index. This cell validates that the residual tensor has at least one row per fine-MCTS JSONL row and records the encoder-input shape that training will see.

In [5]:
fine_pivot_residuals = torch.load(fine_pivot_residuals_path, map_location="cpu", weights_only=True).float()
print("pivot residuals:", shape_of(fine_pivot_residuals))
assert fine_pivot_residuals.shape[0] >= fine_n_rows

if fine_full_residuals_path.exists():
    fine_full_payload = torch.load(fine_full_residuals_path, map_location="cpu", weights_only=False)
    print("full residual payload:", shape_of(fine_full_payload))
else:
    print("full residuals absent; pivot residuals are sufficient for last-token/default compositional training.")

first_residual = fine_pivot_residuals[int(fine_first.get("question_id", 0))]
print("first question_id:", fine_first.get("question_id"), "residual shape:", tuple(first_residual.shape))

pivot residuals: {'shape': (9741, 896), 'dtype': 'torch.float32', 'device': 'cpu'}
full residual payload: {'residuals': {'type': 'list', 'len': 9741, 'first_shape': {'shape': (82, 896), 'dtype': 'torch.float32', 'device': 'cpu'}}, 'lengths': {'shape': (9741,), 'dtype': 'torch.int64', 'device': 'cpu'}}
first question_id: 0 residual shape: (896,)


## 6. Single-row canonicalization

This is the first genuine compositional transformation. Each explored route sequence is converted to a conservative canonical edit program over the anchor sequence. Routes that cannot be represented within `K`, `swap_radius`, and `editable_start` are omitted and counted as unreachable.

In [6]:
from core.edit_dsl import apply_program, canonical_key_str, program_from_dicts
from data_prep.compositional.canonicalize import _canonicalize_row, _editable_indices_for_anchor

K = int(fine_config.get("max_local_edits", 2))
R = int(fine_config.get("swap_radius", 2))
S = int(fine_config.get("editable_start", 0))
include_assign = False
editable_indices = _editable_indices_for_anchor(len(fine_first["anchor_sequence"]), S)

canonical_first, first_stats = _canonicalize_row(
    fine_first,
    K=K,
    swap_radius=R,
    editable_indices=editable_indices,
    include_assign=include_assign,
    dedupe_assign_with_struct=False,
)

display_dict("Canonicalized first row, compact", compact_row(canonical_first, max_programs=12))
display_dict("Canonicalization stats for first row", dict(first_stats))

assert canonical_first["programs"][0]["program_key"] == "noop"
assert canonical_first["programs"][0]["program"] == []
assert canonical_first["n_programs"] == len(canonical_first["programs"])
assert canonical_first["n_unreachable"] == first_stats.get("unreachable", 0)
assert all(len(p["program"]) <= K for p in canonical_first["programs"])

anchor = canonical_first["anchor_sequence"]
for entry in canonical_first["programs"][:20]:
    prog = program_from_dicts(entry["program"])
    assert canonical_key_str(prog) == entry["program_key"]
    out_seq = apply_program(anchor, prog)
    assert len(out_seq) == len(anchor), "compositional edit programs preserve sequence length here"

source_counts = Counter(p["source"] for p in canonical_first["programs"])
length_counts = Counter(p["length"] for p in canonical_first["programs"])
display_dict("First-row canonical source/length counts", {
    "source_counts": dict(source_counts),
    "length_counts": dict(length_counts),
    "program_keys": [p["program_key"] for p in canonical_first["programs"][:20]],
})

### Canonicalized first row, compact

{'benchmark_id': 'commonsenseqa',
 'question_id': 0,
 'question_hash': '0d118b641f7967725f863abf30f869a8',
 'anchor_sequence': [0,
                     1,
                     2,
                     3,
                     4,
                     5,
                     6,
                     7,
                     8,
                     9,
                     10,
                     11,
                     12,
                     13,
                     14,
                     15,
                     16,
                     17,
                     18,
                     17,
                     20,
                     21,
                     22,
                     22],
 'pivot_layer_index': 16,
 'gate_label': 1,
 'anchor_program': [],
 'programs': [{'program': [],
               'program_key': 'noop',
               'length': 0,
               'source': 'anchor',
               'delta': 0.0,
               'score': -4.08203125,
               'router_target_mass': 0

### Canonicalization stats for first row

{'in:anchor': 1,
 'in': 155,
 'canonical:anchor': 1,
 'canonical': 19,
 'in:explored': 153,
 'canonical:explored': 18,
 'unreachable:explored': 135,
 'unreachable': 136,
 'in:best_seq': 1,
 'unreachable:best_seq': 1}


### First-row canonical source/length counts

{'source_counts': {'anchor': 1, 'explored': 18},
 'length_counts': {0: 1, 1: 12, 2: 6},
 'program_keys': ['noop',
                  'repeat(19)',
                  'skip(19)',
                  'repeat(17)+repeat(19)',
                  'repeat(17)',
                  'swap(18,19)',
                  'skip(17)',
                  'repeat(21)',
                  'swap(18,21)',
                  'swap(17,18)',
                  'swap(20,23)',
                  'skip(20)',
                  'skip(19)+skip(20)',
                  'skip(21)',
                  'skip(17)+skip(21)',
                  'skip(17)+repeat(21)',
                  'skip(23)',
                  'skip(17)+repeat(20)',
                  'skip(17)+skip(20)']}


## 7. Directory-level canonicalization

The real pipeline runs the same canonicalization over every benchmark JSONL row and also links/copies residual tensors. This cell rebuilds the canonical CSQA directory under scratch when `RUN_FULL_PIPELINE` is enabled; otherwise it loads the existing canonical summary.

In [7]:
from data_prep.compositional.canonicalize import canonicalize_directory

if RUN_FULL_PIPELINE:
    canonical_summary = canonicalize_directory(
        FINE_MCTS_DIR,
        SCRATCH_CANONICAL_DIR,
        cli_K=K,
        cli_radius=R,
        cli_editable_start=S,
        include_assign=include_assign,
        dedupe_assign_with_struct=False,
        copy_residuals=False,
        benchmarks=[BENCH],
    )
else:
    canonical_summary = json.loads((CANONICAL_DIR / "canonicalization_summary.json").read_text())

canonical_jsonl = CANONICAL_DIR / f"{BENCH}.jsonl"
canonical_config_path = CANONICAL_DIR / "config.json"
canonical_pivot_residuals_path = CANONICAL_DIR / f"{BENCH}_pivot_residuals.pt"

assert canonical_jsonl.is_file()
assert canonical_config_path.is_file()
assert canonical_pivot_residuals_path.exists()
assert count_jsonl(canonical_jsonl) == fine_n_rows

display_dict("Canonicalization summary", canonical_summary)

bench_summary = canonical_summary["benchmarks"][BENCH]
stats = bench_summary["stats"]
length_hist = bench_summary["length_histogram"]
noop_count = int(length_hist.get("0", length_hist.get(0, 0)))
assert bench_summary["rows_in"] == fine_n_rows
assert bench_summary["rows_out"] == fine_n_rows
assert stats["canonical"] + stats["unreachable"] == stats["in"]
assert noop_count == fine_n_rows

### Canonicalization summary

{'benchmarks': {'commonsenseqa': {'rows_in': 9287,
                                  'rows_out': 9287,
                                  'stats': {'in:anchor': 9287,
                                            'in': 1720866,
                                            'canonical:anchor': 9287,
                                            'canonical': 198852,
                                            'in:explored': 1702586,
                                            'canonical:explored': 189565,
                                            'unreachable:explored': 1513021,
                                            'unreachable': 1522014,
                                            'in:best_seq': 8993,
                                            'unreachable:best_seq': 8993},
                                  'length_histogram': {0: 9287,
                                                       1: 93398,
                                                       2: 96167},
                  

## 8. Canonical JSONL schema and distributions

Canonical rows are the source for every downstream compositional artifact. The crucial field is `programs`: a list of canonical programs with `program_key`, `length`, `source`, `delta`, and `score` metadata.

In [8]:
canonical_sample_rows = read_jsonl(canonical_jsonl, limit=512)
canonical_first_from_file = canonical_sample_rows[0]
display_dict("Canonical first row from file, compact", compact_row(canonical_first_from_file, max_programs=10))

n_programs_per_row = [len(r.get("programs", [])) for r in canonical_sample_rows]
unreachable_per_row = [int(r.get("n_unreachable", 0)) for r in canonical_sample_rows]
program_lengths = Counter()
program_sources = Counter()
program_key_counts = Counter()
for row in canonical_sample_rows:
    for entry in row.get("programs", []):
        program_lengths[int(entry["length"])] += 1
        program_sources[entry["source"]] += 1
        program_key_counts[entry["program_key"]] += 1

display_dict("Canonical sample diagnostics", {
    "sample_rows": len(canonical_sample_rows),
    "n_programs_per_row": summarize_numeric(n_programs_per_row),
    "n_unreachable_per_row": summarize_numeric(unreachable_per_row),
    "program_lengths": dict(sorted(program_lengths.items())),
    "program_sources": dict(program_sources),
    "most_common_program_keys": program_key_counts.most_common(15),
})

for row in canonical_sample_rows:
    assert row["programs"][0]["program_key"] == "noop"
    assert row["programs"][0]["length"] == 0
    assert row["n_programs"] == len(row["programs"])
    assert all(len(entry["program"]) == int(entry["length"]) for entry in row["programs"])
    assert all(int(entry["length"]) <= K for entry in row["programs"])

# The file output for the first row should match the direct one-row transform above.
assert canonical_first_from_file["programs"] == canonical_first["programs"]

### Canonical first row from file, compact

{'benchmark_id': 'commonsenseqa',
 'question_id': 0,
 'question_hash': '0d118b641f7967725f863abf30f869a8',
 'anchor_sequence': [0,
                     1,
                     2,
                     3,
                     4,
                     5,
                     6,
                     7,
                     8,
                     9,
                     10,
                     11,
                     12,
                     13,
                     14,
                     15,
                     16,
                     17,
                     18,
                     17,
                     20,
                     21,
                     22,
                     22],
 'pivot_layer_index': 16,
 'gate_label': 1,
 'anchor_program': [],
 'programs': [{'program': [],
               'program_key': 'noop',
               'length': 0,
               'source': 'anchor',
               'delta': 0.0,
               'score': -4.08203125,
               'router_target_mass': 0

### Canonical sample diagnostics

{'sample_rows': 512,
 'n_programs_per_row': {'n': 512,
                        'min': 9.0,
                        'p25': 18.0,
                        'median': 21.0,
                        'p75': 25.0,
                        'max': 32.0,
                        'mean': 21.33203125},
 'n_unreachable_per_row': {'n': 512,
                           'min': 136.0,
                           'p25': 158.0,
                           'median': 165.0,
                           'p75': 170.0,
                           'max': 187.0,
                           'mean': 164.09375},
 'program_lengths': {0: 512, 1: 5149, 2: 5261},
 'program_sources': {'anchor': 512, 'explored': 10410},
 'most_common_program_keys': [('noop', 512),
                              ('skip(17)', 398),
                              ('skip(19)', 384),
                              ('skip(23)', 384),
                              ('repeat(18)', 376),
                              ('repeat(20)', 339),
                      

## 9. Support aggregation

Support tables count which primitives and primitive pairs appear in canonical programs. These tables determine which primitives survive into `O_train`. The no-op program is ignored because it appears in every row and carries no primitive.

In [9]:
from data_prep.compositional.support import aggregate_support, write_support_tables

if RUN_FULL_PIPELINE:
    support_summary = write_support_tables(
        CANONICAL_DIR,
        benchmarks=[BENCH],
        output_dir=CANONICAL_DIR,
        count_unique_per_question=USE_UNIQUE_SUPPORT_PER_QUESTION,
    )
else:
    support_summary = json.loads((CANONICAL_DIR / "support_summary.json").read_text())

primitive_support_path = CANONICAL_DIR / "primitive_support.jsonl"
pair_support_path = CANONICAL_DIR / "pair_support.jsonl"
primitive_support_rows = read_jsonl(primitive_support_path)
pair_support_rows = read_jsonl(pair_support_path)

display_dict("Support summary", support_summary)
display_dict("Top primitive support rows", {"rows": primitive_support_rows[:12]})
display_dict("Top pair support rows", {"rows": pair_support_rows[:8]})

assert primitive_support_rows, "primitive support should not be empty"
assert pair_support_rows, "pair support should not be empty for K=2 CSQA"
assert all(row["count"] > 0 for row in primitive_support_rows)
assert all(row["n_questions"] > 0 for row in primitive_support_rows)
assert all(row["n_benchmarks"] >= 1 for row in primitive_support_rows)

primitive_kind_counts = Counter(row["kind"] for row in primitive_support_rows)
display_dict("Primitive support by kind", dict(primitive_kind_counts))

### Support summary

{'n_primitives': 23,
 'n_pairs': 52,
 'total_primitive_occurrences': 285732,
 'total_pair_occurrences': 96167}


### Top primitive support rows

{'rows': [{'key': 'skip(17)',
           'kind': 'skip',
           'args': [17],
           'count': 32686,
           'n_questions': 8955,
           'n_benchmarks': 1},
          {'key': 'repeat(17)',
           'kind': 'repeat',
           'args': [17],
           'count': 17592,
           'n_questions': 7884,
           'n_benchmarks': 1},
          {'key': 'swap(17,18)',
           'kind': 'swap',
           'args': [17, 18],
           'count': 2840,
           'n_questions': 2840,
           'n_benchmarks': 1},
          {'key': 'swap(17,20)',
           'kind': 'swap',
           'args': [17, 20],
           'count': 2070,
           'n_questions': 2070,
           'n_benchmarks': 1},
          {'key': 'skip(18)',
           'kind': 'skip',
           'args': [18],
           'count': 21567,
           'n_questions': 8236,
           'n_benchmarks': 1},
          {'key': 'repeat(18)',
           'kind': 'repeat',
           'args': [18],
           'count': 22666,
           

### Top pair support rows

{'rows': [{'keys': ['skip(17)', 'skip(18)'],
           'primitives': [{'kind': 'skip', 'args': [17]},
                          {'kind': 'skip', 'args': [18]}],
           'count': 2834,
           'n_questions': 2834,
           'n_benchmarks': 1},
          {'keys': ['skip(17)', 'skip(19)'],
           'primitives': [{'kind': 'skip', 'args': [17]},
                          {'kind': 'skip', 'args': [19]}],
           'count': 3269,
           'n_questions': 3269,
           'n_benchmarks': 1},
          {'keys': ['skip(17)', 'skip(20)'],
           'primitives': [{'kind': 'skip', 'args': [17]},
                          {'kind': 'skip', 'args': [20]}],
           'count': 1902,
           'n_questions': 1902,
           'n_benchmarks': 1},
          {'keys': ['skip(17)', 'skip(21)'],
           'primitives': [{'kind': 'skip', 'args': [17]},
                          {'kind': 'skip', 'args': [21]}],
           'count': 2181,
           'n_questions': 2181,
           'n_benchmarks': 

### Primitive support by kind

{'skip': 7, 'repeat': 5, 'swap': 11}


## 10. Primitive catalogue filtering (`O_train`)

`filter_primitive_catalogue` applies the configured support thresholds and kind filter. The output order is the canonical primitive order used everywhere else: `primitives.jsonl` row `idx == j` is the primitive-score coordinate `u_q[j]` in training.

In [10]:
from core.edit_dsl import prim_key
from data_prep.compositional.catalogue import filter_primitive_catalogue

survivors = filter_primitive_catalogue(
    primitive_support_rows,
    min_count=FILTER_MIN_COUNT,
    min_questions=FILTER_MIN_QUESTIONS,
    min_benchmarks=FILTER_MIN_BENCHMARKS,
    keep_kinds=KEEP_KINDS,
)
primitive_key_to_idx = {canonical_key_str((prim,)): idx for idx, (prim, _row) in enumerate(survivors)}

display_dict("Filtered primitive catalogue preview", {
    "n_support_rows": len(primitive_support_rows),
    "n_survivors": len(survivors),
    "thresholds": {
        "min_count": FILTER_MIN_COUNT,
        "min_questions": FILTER_MIN_QUESTIONS,
        "min_benchmarks": FILTER_MIN_BENCHMARKS,
        "keep_kinds": KEEP_KINDS,
    },
    "survivors_first_20": [
        {"idx": i, "key": canonical_key_str((prim,)), "support": row}
        for i, (prim, row) in enumerate(survivors[:20])
    ],
})

assert survivors, "no primitives survived filtering"
assert list(primitive_key_to_idx.values()) == list(range(len(survivors)))
assert [prim_key(prim) for prim, _ in survivors] == sorted(prim_key(prim) for prim, _ in survivors)

### Filtered primitive catalogue preview

{'n_support_rows': 23,
 'n_survivors': 23,
 'thresholds': {'min_count': 50,
                'min_questions': 50,
                'min_benchmarks': 1,
                'keep_kinds': ('skip', 'repeat', 'swap')},
 'survivors_first_20': [{'idx': 0,
                         'key': 'skip(17)',
                         'support': {'key': 'skip(17)',
                                     'kind': 'skip',
                                     'args': [17],
                                     'count': 32686,
                                     'n_questions': 8955,
                                     'n_benchmarks': 1}},
                        {'idx': 1,
                         'key': 'repeat(17)',
                         'support': {'key': 'repeat(17)',
                                     'kind': 'repeat',
                                     'args': [17],
                                     'count': 17592,
                                     'n_questions': 7884,
                           

## 11. Build the compositional catalogue directory

`build_catalogues` is the main bridge from canonical JSONL to training artifacts. It writes:

- `primitives.jsonl`: global primitive catalogue `O_train`.
- `legal_programs/{bench}.jsonl`: legal program rows for the benchmark anchor.
- `incidence/{bench}.pt`: sparse `A` and dense lengths `ell`.
- `pair_incidence/{bench}.pt`: optional sparse `B` and pair index.
- `observed/{bench}.jsonl`: one record per question with observed legal-row indices and deltas.
- `manifest.json`: paths and counts consumed by `load_artifacts`.

In [11]:
from data_prep.compositional.catalogue import build_catalogues

if RUN_FULL_PIPELINE:
    manifest = build_catalogues(
        CANONICAL_DIR,
        CATALOGUE_DIR,
        support_path=primitive_support_path,
        benchmarks=[BENCH],
        min_count=FILTER_MIN_COUNT,
        min_questions=FILTER_MIN_QUESTIONS,
        min_benchmarks=FILTER_MIN_BENCHMARKS,
        keep_kinds=KEEP_KINDS,
        geometry_overrides=None,
    )
else:
    manifest = json.loads((CATALOGUE_DIR / "manifest.json").read_text())

display_dict("Compositional manifest", manifest)

assert manifest["M"] == len(survivors) or not RUN_FULL_PIPELINE
assert BENCH in manifest["benchmarks"]
bench_info = manifest["benchmarks"][BENCH]
assert bench_info["n_legal_programs"] > 0
assert bench_info["n_questions_kept"] > 0
assert Path(CATALOGUE_DIR / manifest["primitives_path"]).is_file()
assert Path(CATALOGUE_DIR / bench_info["legal_programs_path"]).is_file()
assert Path(CATALOGUE_DIR / bench_info["incidence_path"]).is_file()
assert Path(CATALOGUE_DIR / bench_info["observed_path"]).is_file()

### Compositional manifest

{'data_dir': '/home/janerik/flexible-test-time-program-selection/notebooks/_compositional_csqa_validation_artifacts/csqa_canonical',
 'output_dir': '/home/janerik/flexible-test-time-program-selection/notebooks/_compositional_csqa_validation_artifacts/csqa_compositional',
 'geometry': {'K': 2,
              'swap_radius': 3,
              'editable_start': 17,
              'include_assign': False,
              'dedupe_assign_with_struct': False},
 'filter': {'min_count': 50,
            'min_questions': 50,
            'min_benchmarks': 1,
            'keep_kinds': ['skip', 'repeat', 'swap'],
            'support_path': '/home/janerik/flexible-test-time-program-selection/notebooks/_compositional_csqa_validation_artifacts/csqa_canonical/primitive_support.jsonl'},
 'M': 23,
 'primitives_path': 'primitives.jsonl',
 'benchmarks': {'commonsenseqa': {'anchor': [0,
                                             1,
                                             2,
                                

## 12. Legal programs and incidence tensor validation

For each legal program row `r`, `primitive_indices` lists the primitive coordinates that compose that program. The sparse incidence matrix `A[r, j] = 1` exactly when primitive `j` is in program `r`. The dense length vector stores `len(program_r)` and is used as a program-length penalty in training.

In [12]:
primitives_path = CATALOGUE_DIR / manifest["primitives_path"]
legal_programs_path = CATALOGUE_DIR / bench_info["legal_programs_path"]
incidence_path = CATALOGUE_DIR / bench_info["incidence_path"]
pair_incidence_path = CATALOGUE_DIR / bench_info["pair_incidence_path"]

primitive_rows = read_jsonl(primitives_path)
legal_rows = read_jsonl(legal_programs_path)
incidence_payload = torch.load(incidence_path, map_location="cpu", weights_only=True)
pair_payload = torch.load(pair_incidence_path, map_location="cpu", weights_only=True)

assert_contiguous_indices(primitive_rows)
assert_contiguous_indices(legal_rows)
assert legal_rows[0]["key"] == "noop"
assert legal_rows[0]["primitive_indices"] == []
assert legal_rows[0]["length"] == 0

A = torch.sparse_coo_tensor(
    incidence_payload["A_indices"].long(),
    incidence_payload["A_values"].float(),
    size=tuple(incidence_payload["A_shape"]),
).coalesce()
lengths = incidence_payload["lengths"].float()
B = torch.sparse_coo_tensor(
    pair_payload["B_indices"].long(),
    pair_payload["B_values"].float(),
    size=tuple(pair_payload["B_shape"]),
).coalesce()
pair_index = pair_payload["pair_index"].long()

print("|O_train| M:", len(primitive_rows))
print("|E_legal| N:", len(legal_rows))
print("A:", A.shape, "nnz=", A._nnz())
print("lengths:", shape_of(lengths))
print("B:", B.shape, "nnz=", B._nnz())
print("pair_index:", shape_of(pair_index))

assert A.shape == (len(legal_rows), len(primitive_rows))
assert lengths.shape == (len(legal_rows),)
assert torch.all(lengths >= 0)
assert pair_index.shape[1] == 2 if pair_index.numel() else True

# Spot-check all rows here; CSQA has a small legal catalogue.
A_dense = A.to_dense()
for row in legal_rows:
    r = int(row["idx"])
    prims = sorted(int(x) for x in row["primitive_indices"])
    nz = sorted(A_dense[r].nonzero(as_tuple=False).flatten().tolist())
    assert nz == prims, f"A row {r} does not match primitive_indices"
    assert int(lengths[r].item()) == int(row["length"])
    assert len(prims) == int(row["length"])

display_dict("Legal-program preview", {"rows": legal_rows[:20]})
display_dict("Legal-program length histogram", dict(Counter(int(r["length"]) for r in legal_rows)))

|O_train| M: 23
|E_legal| N: 179
A: torch.Size([179, 23]) nnz= 333
lengths: {'shape': (179,), 'dtype': 'torch.float32', 'device': 'cpu'}
B: torch.Size([179, 155]) nnz= 155
pair_index: {'shape': (155, 2), 'dtype': 'torch.int64', 'device': 'cpu'}


### Legal-program preview

{'rows': [{'idx': 0, 'length': 0, 'primitive_indices': [], 'key': 'noop'},
          {'idx': 1, 'length': 1, 'primitive_indices': [0], 'key': 'skip(17)'},
          {'idx': 2,
           'length': 1,
           'primitive_indices': [1],
           'key': 'repeat(17)'},
          {'idx': 3,
           'length': 1,
           'primitive_indices': [2],
           'key': 'swap(17,18)'},
          {'idx': 4,
           'length': 1,
           'primitive_indices': [3],
           'key': 'swap(17,20)'},
          {'idx': 5, 'length': 1, 'primitive_indices': [4], 'key': 'skip(18)'},
          {'idx': 6,
           'length': 1,
           'primitive_indices': [5],
           'key': 'repeat(18)'},
          {'idx': 7,
           'length': 1,
           'primitive_indices': [6],
           'key': 'swap(18,19)'},
          {'idx': 8,
           'length': 1,
           'primitive_indices': [7],
           'key': 'swap(18,20)'},
          {'idx': 9,
           'length': 1,
           'primitive_indi

### Legal-program length histogram

{0: 1, 1: 23, 2: 155}


## 13. Pair-incidence validation

When pair scoring is enabled, `pair_index[p] = (i, j)` names a pair of primitive coordinates and `B[r, p] = 1` if legal program `r` contains both primitives. This is optional in the model, but the catalogue builder emits it for diagnostics and pairwise training.

In [13]:
B_dense = B.to_dense()
pair_tuples = [tuple(map(int, row.tolist())) for row in pair_index]
assert len(pair_tuples) == len(set(pair_tuples)), "pair_index should have unique rows"
assert all(i < j for i, j in pair_tuples), "pair_index rows should be ordered i < j"

pair_to_col = {pair: p for p, pair in enumerate(pair_tuples)}
for row in legal_rows:
    r = int(row["idx"])
    prims = sorted(int(x) for x in row["primitive_indices"])
    expected_pairs = []
    for a_pos in range(len(prims)):
        for b_pos in range(a_pos + 1, len(prims)):
            expected_pairs.append((prims[a_pos], prims[b_pos]))
    expected_cols = sorted(pair_to_col[pair] for pair in expected_pairs)
    got_cols = sorted(B_dense[r].nonzero(as_tuple=False).flatten().tolist())
    assert got_cols == expected_cols, f"B row {r} does not match primitive pair set"

pair_preview = []
for p, (i, j) in enumerate(pair_tuples[:20]):
    pair_preview.append({
        "pair_col": p,
        "i": i,
        "j": j,
        "i_key": primitive_rows[i]["key"],
        "j_key": primitive_rows[j]["key"],
    })
display_dict("Pair-index preview", {"rows": pair_preview})

### Pair-index preview

{'rows': [{'pair_col': 0,
           'i': 0,
           'j': 4,
           'i_key': 'skip(17)',
           'j_key': 'skip(18)'},
          {'pair_col': 1,
           'i': 0,
           'j': 5,
           'i_key': 'skip(17)',
           'j_key': 'repeat(18)'},
          {'pair_col': 2,
           'i': 0,
           'j': 6,
           'i_key': 'skip(17)',
           'j_key': 'swap(18,19)'},
          {'pair_col': 3,
           'i': 0,
           'j': 7,
           'i_key': 'skip(17)',
           'j_key': 'swap(18,20)'},
          {'pair_col': 4,
           'i': 0,
           'j': 8,
           'i_key': 'skip(17)',
           'j_key': 'swap(18,21)'},
          {'pair_col': 5,
           'i': 0,
           'j': 9,
           'i_key': 'skip(17)',
           'j_key': 'skip(19)'},
          {'pair_col': 6,
           'i': 0,
           'j': 10,
           'i_key': 'skip(17)',
           'j_key': 'repeat(19)'},
          {'pair_col': 7,
           'i': 0,
           'j': 11,
           'i_key'

## 14. Observed-candidate projection

`collect_observed_for_benchmark` maps each canonical program entry to its legal-program row. If multiple MCTS entries map to the same canonical program row, their deltas are averaged. This produces the sparse observed view that the default compositional loss uses.

In [14]:
from data_prep.compositional.catalogue import collect_observed_for_benchmark

observed_path = CATALOGUE_DIR / bench_info["observed_path"]
observed_rows = read_jsonl(observed_path)
key_to_row = {str(row["key"]): int(row["idx"]) for row in legal_rows}
recomputed_observed, recomputed_stats = collect_observed_for_benchmark(
    canonical_jsonl,
    key_to_row,
    len(legal_rows),
)

print("observed rows:", len(observed_rows))
print("recomputed observed rows:", len(recomputed_observed))
display_dict("Observed recomputation stats", dict(recomputed_stats))
display_dict("First observed records", {"rows": observed_rows[:5]})

assert observed_rows == recomputed_observed, "observed file should equal collect_observed_for_benchmark output"
assert len(observed_rows) == bench_info["n_questions_kept"]
assert all(int(rec["n_obs"]) == len(rec["obs_indices"]) == len(rec["obs_deltas"]) for rec in observed_rows)
assert all(0 <= int(i) < len(legal_rows) for rec in observed_rows for i in rec["obs_indices"])
assert all(rec["obs_indices"] == sorted(rec["obs_indices"]) for rec in observed_rows)

n_obs_values = [int(rec["n_obs"]) for rec in observed_rows]
all_obs_deltas = [float(d) for rec in observed_rows for d in rec["obs_deltas"]]
positive_obs_counts = [sum(1 for d in rec["obs_deltas"] if float(d) > 0.0) for rec in observed_rows]
display_dict("Observed-candidate diagnostics", {
    "n_obs_per_question": summarize_numeric(n_obs_values),
    "positive_obs_per_question": summarize_numeric(positive_obs_counts),
    "observed_delta_summary": summarize_numeric(all_obs_deltas[:200000]),
    "top_observed_program_rows": Counter(i for rec in observed_rows for i in rec["obs_indices"]).most_common(20),
})

observed rows: 9287
recomputed observed rows: 9287


### Observed recomputation stats

{'rows_in': 9287,
 'rows_kept': 9287,
 'observed_pairs': 198852,
 'dropped_program_entries': 0}


### First observed records

{'rows': [{'residual_idx': 0,
           'question_id': 0,
           'question_hash': '0d118b641f7967725f863abf30f869a8',
           'n_obs': 19,
           'obs_indices': [0,
                           1,
                           2,
                           3,
                           7,
                           9,
                           10,
                           11,
                           13,
                           17,
                           18,
                           19,
                           23,
                           32,
                           33,
                           37,
                           38,
                           44,
                           133],
           'obs_deltas': [0.0,
                          0.380859375,
                          1.26171875,
                          1.177734375,
                          0.640625,
                          -1.25,
                          -0.3046875,
              

### Observed-candidate diagnostics

{'n_obs_per_question': {'n': 9287,
                        'min': 7.0,
                        'p25': 18.0,
                        'median': 21.0,
                        'p75': 24.0,
                        'max': 37.0,
                        'mean': 21.41186604931625},
 'positive_obs_per_question': {'n': 9287,
                               'min': 0.0,
                               'p25': 5.0,
                               'median': 7.0,
                               'p75': 10.0,
                               'max': 21.0,
                               'mean': 7.551308280391946},
 'observed_delta_summary': {'n': 198852,
                            'min': -11.46142578125,
                            'p25': -0.67578125,
                            'median': -0.1220703125,
                            'p75': 0.125,
                            'max': 4.2001953125,
                            'mean': -0.37645252851653066},
 'top_observed_program_rows': [(0, 9287),
                   

## 15. Artifact loading: exact trainer entry point

Training starts by calling `load_artifacts(catalogue_dir, benchmarks=...)`. This builds typed primitive specs and sparse `LegalCatalogue` objects, and verifies that every benchmark incidence matrix has the same primitive dimension as `primitives.jsonl`.

In [15]:
from routers.compositional_router import load_artifacts, load_legal_programs_jsonl

artifacts = load_artifacts(CATALOGUE_DIR, benchmarks=[BENCH])
cat = artifacts.catalogues[BENCH]

print("loaded benchmarks:", artifacts.benchmarks)
print("num_primitives:", artifacts.num_primitives)
print("num_layers:", artifacts.num_layers)
print("catalogue benchmark:", cat.benchmark)
print("catalogue anchor length:", len(cat.anchor))
print("catalogue A:", cat.A.shape, "nnz=", cat.A._nnz())
print("catalogue lengths:", tuple(cat.lengths.shape))
print("catalogue has pairs:", cat.has_pairs, "n_pairs=", cat.n_pairs)

assert artifacts.num_primitives == len(primitive_rows)
assert cat.n_programs == len(legal_rows)
assert cat.n_primitives == len(primitive_rows)
assert torch.equal(cat.A.cpu().to_dense(), A_dense)
assert torch.equal(cat.lengths.cpu(), lengths)
if cat.has_pairs:
    assert torch.equal(cat.B.cpu().to_dense(), B_dense)
    assert torch.equal(cat.pair_index.cpu(), pair_index)

primitive_preview = [p for p in artifacts.primitives[:10]]
display_dict("PrimitiveSpec preview", {"rows": primitive_preview})

loaded benchmarks: ['commonsenseqa']
num_primitives: 23
num_layers: 24
catalogue benchmark: commonsenseqa
catalogue anchor length: 24
catalogue A: torch.Size([179, 23]) nnz= 333
catalogue lengths: (179,)
catalogue has pairs: True n_pairs= 155


### PrimitiveSpec preview

{'rows': [PrimitiveSpec(idx=0, kind='skip', args=(17,), key='skip(17)'),
          PrimitiveSpec(idx=1, kind='repeat', args=(17,), key='repeat(17)'),
          PrimitiveSpec(idx=2, kind='swap', args=(17, 18), key='swap(17,18)'),
          PrimitiveSpec(idx=3, kind='swap', args=(17, 20), key='swap(17,20)'),
          PrimitiveSpec(idx=4, kind='skip', args=(18,), key='skip(18)'),
          PrimitiveSpec(idx=5, kind='repeat', args=(18,), key='repeat(18)'),
          PrimitiveSpec(idx=6, kind='swap', args=(18, 19), key='swap(18,19)'),
          PrimitiveSpec(idx=7, kind='swap', args=(18, 20), key='swap(18,20)'),
          PrimitiveSpec(idx=8, kind='swap', args=(18, 21), key='swap(18,21)'),
          PrimitiveSpec(idx=9, kind='skip', args=(19,), key='skip(19)')]}


## 16. Dataset construction: training sample structure

`CompositionalDataset` joins three things:

- encoder input from residual tensors,
- observed legal-program indices and deltas from `observed/{bench}.jsonl`,
- benchmark identity so joint training can select the right catalogue.

This is the per-question structure consumed by the PyTorch `DataLoader`.

In [16]:
from routers.compositional_router import CompositionalDataset

bench_to_id = {BENCH: 0}
dataset = CompositionalDataset(
    artifacts,
    benchmarks=[BENCH],
    use_full_sequence=False,
    bench_to_id=bench_to_id,
)

print("dataset length:", len(dataset))
assert len(dataset) == len(observed_rows)

sample = dataset[0]
print("sample keys:", sorted(sample.keys()))
print("encoder_input:", shape_of(sample["encoder_input"]))
print("obs_indices:", shape_of(sample["obs_indices"]), sample["obs_indices"][:10].tolist())
print("obs_deltas:", shape_of(sample["obs_deltas"]), sample["obs_deltas"][:10].tolist())
print("benchmark_id:", sample["benchmark_id"], "benchmark:", sample["benchmark"], "question_id:", sample["question_id"])

assert sample["benchmark"] == BENCH
assert int(sample["benchmark_id"]) == bench_to_id[BENCH]
assert torch.equal(sample["obs_indices"], torch.tensor(observed_rows[0]["obs_indices"], dtype=torch.long))
assert torch.allclose(sample["obs_deltas"], torch.tensor(observed_rows[0]["obs_deltas"], dtype=torch.float32))
assert sample["encoder_input"].shape == fine_pivot_residuals[0].shape

dataset length: 9287
sample keys: ['benchmark', 'benchmark_id', 'encoder_input', 'obs_deltas', 'obs_indices', 'question_id']
encoder_input: {'shape': (896,), 'dtype': 'torch.float32', 'device': 'cpu'}
obs_indices: {'shape': (19,), 'dtype': 'torch.int64', 'device': 'cpu'} [0, 1, 2, 3, 7, 9, 10, 11, 13, 17]
obs_deltas: {'shape': (19,), 'dtype': 'torch.float32', 'device': 'cpu'} [0.0, 0.380859375, 1.26171875, 1.177734375, 0.640625, -1.25, -0.3046875, 1.57421875, 1.740234375, 0.125]
benchmark_id: 0 benchmark: commonsenseqa question_id: 0


## 17. Collation: batch structure used by the loss

The collator pads each row's observed candidates to the maximum `n_obs` in the batch and builds `obs_mask`. The loss later gathers scores at `obs_indices` and normalizes only over positions where `obs_mask == 1`.

In [17]:
from torch.utils.data import DataLoader
from routers.compositional_router import collate_compositional, softmax_ce_on_observed

loader = DataLoader(dataset, batch_size=8, shuffle=False, collate_fn=collate_compositional)
batch = next(iter(loader))

for key, value in batch.items():
    print(key, "=>", shape_of(value))

assert batch["encoder_input"].shape[0] == 8
assert batch["obs_indices"].shape == batch["obs_deltas"].shape == batch["obs_mask"].shape
assert torch.all(batch["obs_mask"].sum(dim=1) > 0)
assert torch.all(batch["obs_indices"][batch["obs_mask"].bool()] >= 0)
assert torch.all(batch["obs_indices"][batch["obs_mask"].bool()] < cat.n_programs)

# Demonstrate the exact score-gathering shape used by the trainer without instantiating a router.
# In training, u_q comes from the router's primitive scorer. Here we use deterministic dummy scores.
Bsz = batch["encoder_input"].shape[0]
u_q = torch.linspace(-1.0, 1.0, steps=cat.n_primitives).unsqueeze(0).repeat(Bsz, 1)
program_scores = torch.sparse.mm(cat.A, u_q.t()).t() - 0.01 * cat.lengths.unsqueeze(0)
gathered = program_scores.gather(1, batch["obs_indices"].clamp(min=0))
ce = softmax_ce_on_observed(
    gathered,
    torch.arange(gathered.shape[1]).unsqueeze(0).expand_as(gathered),
    batch["obs_deltas"],
    batch["obs_mask"],
    tau=1.0,
    student_temp=1.0,
)

print("dummy u_q:", tuple(u_q.shape))
print("program_scores = A @ u_q - lambda*lengths:", tuple(program_scores.shape))
print("gathered observed scores:", tuple(gathered.shape))
print("softmax CE on observed candidates:", float(ce.item()))
assert gathered.shape == batch["obs_indices"].shape
assert torch.isfinite(ce)

encoder_input => {'shape': (8, 896), 'dtype': 'torch.float32', 'device': 'cpu'}
attention_mask => NoneType
benchmark_id => {'shape': (8,), 'dtype': 'torch.int64', 'device': 'cpu'}
benchmark => {'type': 'list', 'len': 8, 'first_shape': 'str'}
question_id => {'shape': (8,), 'dtype': 'torch.int64', 'device': 'cpu'}
obs_indices => {'shape': (8, 24), 'dtype': 'torch.int64', 'device': 'cpu'}
obs_deltas => {'shape': (8, 24), 'dtype': 'torch.float32', 'device': 'cpu'}
obs_mask => {'shape': (8, 24), 'dtype': 'torch.float32', 'device': 'cpu'}
dummy u_q: (8, 23)
program_scores = A @ u_q - lambda*lengths: (8, 179)
gathered observed scores: (8, 24)
softmax CE on observed candidates: 3.181657314300537


## 18. Optional dense-delta materialization

The default compositional path trains on sparse observed candidates. Some experiments use dense supervision controls. `build_dense` projects canonical program deltas into a full `[Q, N_legal]` matrix, filling unobserved legal programs with a large negative value. This section validates that optional artifact and shows how `CompositionalDataset` attaches it.

In [18]:
from data_prep.compositional.dense_from_canonical import build_dense

BUILD_DENSE_FROM_CANONICAL = os.environ.get("BUILD_DENSE_FROM_CANONICAL", "1") != "0"
if BUILD_DENSE_FROM_CANONICAL:
    dense_payload = build_dense(legal_programs_path, canonical_jsonl, missing_fill=-1e3)
    torch.save(dense_payload, SCRATCH_DENSE_PT)
else:
    dense_payload = torch.load(SCRATCH_DENSE_PT, map_location="cpu", weights_only=True)

print("dense output path:", SCRATCH_DENSE_PT)
print("dense payload:", shape_of(dense_payload))
assert dense_payload["delta_matrix"].shape == (fine_n_rows, len(legal_rows))
assert dense_payload["anchor_utilities"].shape == (fine_n_rows,)
assert torch.all(torch.isfinite(dense_payload["anchor_utilities"]))

# The no-op column is row 0 by construction; its delta should be 0 where present.
noop_col = key_to_row["noop"]
assert noop_col == 0
assert torch.allclose(dense_payload["delta_matrix"][:, noop_col], torch.zeros(fine_n_rows))

# Dense-aware dataset attaches dense_deltas and anchor_utility per sample.
dense_dataset = CompositionalDataset(
    artifacts,
    benchmarks=[BENCH],
    use_full_sequence=False,
    bench_to_id=bench_to_id,
    dense_delta_paths={BENCH: SCRATCH_DENSE_PT},
)
dense_sample = dense_dataset[0]
print("dense sample keys:", sorted(dense_sample.keys()))
print("dense_deltas:", shape_of(dense_sample["dense_deltas"]))
print("anchor_utility:", dense_sample["anchor_utility"])
assert dense_sample["dense_deltas"].shape == (cat.n_programs,)

dense output path: /home/janerik/flexible-test-time-program-selection/notebooks/_compositional_csqa_validation_artifacts/commonsenseqa_dense_from_canonical.pt
dense payload: {'delta_matrix': {'shape': (9287, 179), 'dtype': 'torch.float32', 'device': 'cpu'}, 'anchor_utilities': {'shape': (9287,), 'dtype': 'torch.float32', 'device': 'cpu'}}
dense sample keys: ['anchor_utility', 'benchmark', 'benchmark_id', 'dense_deltas', 'encoder_input', 'obs_deltas', 'obs_indices', 'question_id']
dense_deltas: {'shape': (179,), 'dtype': 'torch.float32', 'device': 'cpu'}
anchor_utility: -4.08203125


## 19. End-to-end contract summary

If every assertion above passes, the CSQA compositional data contract is intact:

- fine MCTS rows have aligned route candidates and residual rows;
- canonicalization maps reachable routes to bounded edit programs and drops unreachable routes explicitly;
- support aggregation and filtering produce the primitive coordinate system `O_train`;
- legal-program enumeration produces the row coordinate system `E_legal`;
- sparse `A`, optional sparse `B`, and `lengths` match the JSONL catalogue exactly;
- observed records point only to valid legal rows and preserve per-question deltas;
- `load_artifacts`, `CompositionalDataset`, and `collate_compositional` produce the tensors used by the training loss.

The final training batch fields are:

- `encoder_input`: question residual representation;
- `attention_mask`: optional sequence mask for full-sequence compressors;
- `benchmark_id` / `benchmark`: benchmark routing metadata;
- `question_id`: dense-target row id when dense supervision is used;
- `obs_indices`: legal-program rows observed for each question;
- `obs_deltas`: utility deltas for those observed rows;
- `obs_mask`: padding mask over observed candidates;
- optional `dense_deltas`, `dense_keep_mask`, `anchor_utility`, and local Möbius targets when those experiment modes are enabled.